In [ ]:
import os
os.chdir(os.path.join(os.path.dirname("__file__"), ".."))
import sys
sys.path.insert(0, "..")
import json
import matplotlib.pyplot as plt
from src.data.synthetic import generate_dataset
from src.pipeline.config import PipelineConfig
from src.pipeline.pipeline import run

config = PipelineConfig(ocr_model="tesseract", confidence_threshold=0.3, max_sequence_length=128)

In [ ]:
test_docs = generate_dataset(n=10, seed=99)
results = [run(doc.image, config) for doc in test_docs]
print(f"Processed {len(results)} documents")

In [ ]:
for i, (doc, result) in enumerate(zip(test_docs[:3], results[:3])):
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].imshow(doc.image)
    axes[0].set_title(f"Document {i+1}")
    axes[0].axis("off")
    entity_text = "\n".join(f"{e.label}: {e.value} ({e.confidence:.2f})" for e in result.entities)
    axes[1].text(0.05, 0.5, f"OCR:\n{result.raw_text[:80]}\n\nEntities:\n{entity_text}",
                 transform=axes[1].transAxes, fontsize=9, va="center", family="monospace")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
correct = 0
total = 0
for doc, result in zip(test_docs, results):
    gt = {e.label: e.value for e in doc.entities}
    pred = {e.label: e.value for e in result.entities}
    for label in gt:
        total += 1
        if label in pred and pred[label].lower() in doc.text.lower():
            correct += 1
accuracy = correct / total if total > 0 else 0.0
print(f"Entity label hit rate: {correct}/{total} = {accuracy:.1%}")

In [ ]:
sample = results[0]
output = {
    "raw_text": sample.raw_text,
    "ocr_confidence": round(sample.ocr_confidence, 3),
    "model_used": sample.model_used,
    "entities": [
        {"label": e.label, "value": e.value, "confidence": round(e.confidence, 3)}
        for e in sample.entities
    ],
}
print(json.dumps(output, indent=2))